# ORACLE: Attractor Landscape Analysis

Map the epigenetic landscape using Boolean network simulation and continuous ODE dynamics.
Identify cancer and normal attractors, compute basin sizes, and visualize the energy landscape.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import anndata as ad
import pickle
from pathlib import Path

DATA_DIR = Path('../data')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Preprocessed Data and GRN

In [ ]:
adata = ad.read_h5ad(DATA_DIR / 'processed/anndata/colorectal_GSE132465_processed.h5ad')
with open(DATA_DIR / 'processed/networks/colorectal_GSE132465_grn.pkl', 'rb') as f:
    grn = pickle.load(f)

print(f'AnnData: {adata.n_obs} cells x {adata.n_vars} genes')
print(f'GRN: {grn.number_of_nodes()} nodes, {grn.number_of_edges()} edges')

## 2. Boolean Network Simulation

In [ ]:
from oracle.cam.boolean_network import BooleanNetworkSimulator
from oracle.cam.preprocessing import CAMConfig

config = CAMConfig(n_attractor_samples=5000, n_basin_samples=20000)
bool_net = BooleanNetworkSimulator(grn, config)

print('Running Boolean network simulation...')
bool_attractors = bool_net.find_attractors(n_initial_states=config.n_attractor_samples)
print(f'Found {len(bool_attractors)} Boolean attractors')

basin_sizes = bool_net.compute_basin_sizes(bool_attractors, n_samples=config.n_basin_samples)
print('Basin size distribution:')
for i, (size) in enumerate(basin_sizes.values()):
    print(f'  Attractor {i}: {size:.1%} of state space')

## 3. Continuous ODE Dynamics

In [ ]:
from oracle.cam.continuous_ode import ContinuousGRNDynamics

ode_model = ContinuousGRNDynamics(grn, config).to(device)
print('Finding continuous fixed points...')
continuous_attractors = ode_model.find_fixed_points(n_init=500)
print(f'Found {len(continuous_attractors)} continuous fixed points')

## 4. Attractor Classification

In [ ]:
from oracle.cam.attractor_classifier import AttractorClassifier

classifier = AttractorClassifier(cancer_type='colorectal', tissue='colon')
genes = list(grn.nodes())
labels = classifier.classify(bool_attractors, genes)

print('Attractor classification:')
for i, (attractor, label) in enumerate(zip(bool_attractors, labels)):
    score = classifier._compute_cancer_score(attractor, genes)
    print(f'  Attractor {i}: {label} (cancer score={score:.3f})')

cancer_attr, normal_attr = classifier.get_cancer_normal_pair(bool_attractors, labels)
print(f'\nCancer attractor: {cancer_attr is not None}')
print(f'Normal attractor: {normal_attr is not None}')

## 5. Landscape Visualization

In [ ]:
from oracle.cam.landscape_computer import LandscapeComputer

lc = LandscapeComputer(config)
landscape = lc.compute(adata, bool_net, ode_model, bool_attractors, labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# UMAP with cell states
colors = {'normal': '#2196F3', 'cancer': '#F44336', 'transitional': '#FF9800'}
cell_colors = [colors.get(s, '#9E9E9E') for s in adata.obs.get('cell_state', ['unknown']*adata.n_obs)]
if 'X_umap' in adata.obsm:
    axes[0].scatter(adata.obsm['X_umap'][:, 0], adata.obsm['X_umap'][:, 1],
                    c=cell_colors, s=1, alpha=0.3)
    # Plot attractor positions
    if landscape.get('landscape_embedding') is not None:
        emb = landscape['landscape_embedding']
        for j, (pos, label) in enumerate(zip(emb, labels)):
            color = colors.get(label, 'gray')
            axes[0].scatter(pos[0], pos[1], s=300, c=color, marker='*',
                            edgecolors='black', linewidths=1.5, zorder=5)
axes[0].set_title('Attractor Landscape (UMAP)')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

# Energy surface
if landscape.get('energy_surface') is not None:
    energy = landscape['energy_surface']
    extent = landscape.get('umap_extent', [-1, 1, -1, 1])
    im = axes[1].imshow(energy.T, origin='lower', extent=extent,
                        cmap='RdYlBu_r', aspect='auto')
    plt.colorbar(im, ax=axes[1], label='Waddington Energy')
    axes[1].set_title('Epigenetic Energy Landscape')

plt.tight_layout()
plt.show()

## 6. Pseudotime Analysis

In [ ]:
from oracle.cam.pseudotime import PseudotimeComputer

pt = PseudotimeComputer(config)
adata_pt = pt.compute(adata, cancer_attr, normal_attr)

if 'dpt_pseudotime' in adata_pt.obs:
    sc_import_worked = True
    try:
        import scanpy as sc
        sc.pl.umap(adata_pt, color='dpt_pseudotime', cmap='viridis',
                   title='Pseudotime (normal → cancer trajectory)')
    except Exception:
        sc_import_worked = False
    
    if not sc_import_worked and 'X_umap' in adata_pt.obsm:
        plt.figure(figsize=(8, 6))
        sc_pt = adata_pt.obs['dpt_pseudotime'].values
        plt.scatter(adata_pt.obsm['X_umap'][:, 0], adata_pt.obsm['X_umap'][:, 1],
                    c=sc_pt, cmap='viridis', s=2, alpha=0.5)
        plt.colorbar(label='Pseudotime')
        plt.title('Pseudotime (normal → cancer)')
        plt.show()